In [76]:
import xml.etree.ElementTree as ET
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

/Users/parsa.h/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Read our xml data for one picture

In [44]:
xml_path="../data/train/00001.xml"
tree = ET.parse(xml_path)
root=tree.getroot()

filename = root.find("filename").text
print("Image:", filename)

for obj in root.findall("object"):

    character = obj.find("name").text

    box = obj.find("bndbox")

    xmin = int(box.find("xmin").text)
    ymin = int(box.find("ymin").text)
    xmax = int(box.find("xmax").text)
    ymax = int(box.find("ymax").text)

    print(character, xmin, ymin, xmax, ymax)

Image: for_xml_2811.jpg
1 59 56 101 171
2 96 63 149 181
ز 157 97 211 199
8 219 113 273 216
1 285 124 325 245
6 318 141 382 242
1 396 177 435 282
1 442 192 488 295


load our image (one image for test)

In [45]:
image = cv2.imread("../data/train/00001.jpg")

find characters in our one images

In [46]:
characters = []

for obj in root.findall("object"):

    label = obj.find("name").text

    box = obj.find("bndbox")

    xmin = int(box.find("xmin").text)
    ymin = int(box.find("ymin").text)
    xmax = int(box.find("xmax").text)
    ymax = int(box.find("ymax").text)

    cropped_character = image[ymin:ymax, xmin:xmax]
    characters.append((label, cropped_character))

see our characters in our one image

In [ ]:
for label, char_image in characters:

    char_rgb = cv2.cvtColor(char_image, cv2.COLOR_BGR2RGB)

    plt.imshow(char_rgb)
    plt.title(label)
    plt.axis("off")
    plt.show()

Resizing and gray scaling our one image

In [48]:
target_width = 32
target_height = 64

resized_characters = []

for label, char_image in characters:
    resized_image = cv2.resize(
        char_image,
        (target_width, target_height)
    )
    resized_characters.append((label, resized_image))

grayscale_characters = []

for label , char_image in resized_characters:
    gray_image=cv2.cvtColor(char_image,cv2.COLOR_BGR2GRAY)
    grayscale_characters.append((label, gray_image))

In [ ]:
for label , char_image in grayscale_characters:
    plt.imshow(char_image)
    plt.title(label)
    plt.axis("off")
    plt.show()

Normalization for one image

In [50]:
normalized_characters = []

for label, gray_image in grayscale_characters:
    normalized_image = gray_image.astype("float32") / 255.0
    normalized_characters.append((label, normalized_image))

label, image = normalized_characters[0]



Preprocessing Function for the Full Dataset

In [51]:
def process_plate(image_path, xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    image = cv2.imread(image_path)
    characters = []

    for obj in root.findall("object"):

        label = obj.find("name").text

        box = obj.find("bndbox")

        xmin = int(box.find("xmin").text)
        ymin = int(box.find("ymin").text)
        xmax = int(box.find("xmax").text)
        ymax = int(box.find("ymax").text)

        height, width = image.shape[:2]

        xmin = max(0, min(xmin, width))
        xmax = max(0, min(xmax, width))

        ymin = max(0, min(ymin, height))
        ymax = max(0, min(ymax, height))

        if xmin >= xmax or ymin >= ymax:
            continue
        cropped_character = image[ymin:ymax, xmin:xmax]

        resized_character = cv2.resize(cropped_character,(32, 64))

        gray_character = cv2.cvtColor(resized_character,cv2.COLOR_BGR2GRAY)

        normalized_character = (
            gray_character.astype("float32") / 255.0
        )

        characters.append((label, normalized_character))

    return characters

In [52]:
train_path = Path("../data/train")
xml_files = sorted(train_path.glob("*.xml"))
print("Number of XML files:", len(xml_files))

Number of XML files: 19381


Put our training data into our training dataset

In [53]:
X_train = []
y_train = []

for xml_path in xml_files:

    image_path = xml_path.with_suffix(".jpg") # find the .jpg image that has a same name as xml_path

    characters = process_plate(
        str(image_path),
        str(xml_path)
    )

    for label, image in characters:
        X_train.append(image)
        y_train.append(label)

One problem in my dataset

In [54]:
xml_path = "../data/train/01537.xml"
image_path = "../data/train/01537.jpg"

tree = ET.parse(xml_path)
root = tree.getroot()

image = cv2.imread(image_path)

print("Image shape:", image.shape)

for obj in root.findall("object"):
    label = obj.find("name").text

    box = obj.find("bndbox")

    xmin = int(box.find("xmin").text)
    ymin = int(box.find("ymin").text)
    xmax = int(box.find("xmax").text)
    ymax = int(box.find("ymax").text)

    cropped = image[ymin:ymax, xmin:xmax]

    print(
        "Label:", label,
        "Box:", xmin, ymin, xmax, ymax,
        "Crop shape:", cropped.shape
    )

Image shape: (175, 242, 3)
Label: 1 Box: 508 1355 571 1488 Crop shape: (0, 0, 3)
Label: 3 Box: 566 1374 623 1516 Crop shape: (0, 0, 3)
Label: ت Box: 623 1444 715 1526 Crop shape: (0, 0, 3)
Label: 8 Box: 710 1420 771 1548 Crop shape: (0, 0, 3)
Label: 8 Box: 777 1444 823 1562 Crop shape: (0, 0, 3)
Label: 8 Box: 830 1448 883 1585 Crop shape: (0, 0, 3)
Label: 1 Box: 894 1484 948 1617 Crop shape: (0, 0, 3)
Label: 1 Box: 951 1499 1023 1626 Crop shape: (0, 0, 3)
Label: 4 Box: 21 49 42 101 Crop shape: (52, 21, 3)
Label: 8 Box: 38 49 57 100 Crop shape: (51, 19, 3)
Label: د Box: 59 53 78 104 Crop shape: (51, 19, 3)
Label: 2 Box: 79 53 95 115 Crop shape: (62, 16, 3)
Label: 9 Box: 97 58 115 116 Crop shape: (58, 18, 3)
Label: 9 Box: 109 55 130 113 Crop shape: (58, 21, 3)
Label: 7 Box: 131 66 150 123 Crop shape: (57, 19, 3)
Label: 7 Box: 143 68 173 120 Crop shape: (52, 30, 3)


Converting our training dataset to numpy array

In [55]:
X_train = np.array(X_train)
y_train = np.array(y_train)

Seeing how many characters do we have and how many classes

In [56]:
unique_labels = np.unique(y_train)
print("Labels:", unique_labels)
print("Number of classes:", len(unique_labels))
labels, counts = np.unique(
    y_train,
    return_counts=True
)

for label, count in zip(labels, counts):
    print(label, count)

Labels: ['0' '1' '2' '3' '4' '5' '6' '7' '8' '9' 'الف' 'ب' 'ت' 'ث' 'ج' 'د' 'ز' 'س'
 'ش' 'ص' 'ط' 'ع' 'ق' 'ل' 'م' 'ن' 'ه\u200d' 'و' 'پ'
 'ژ (معلولین و جانبازان)' 'ی']
Number of classes: 31
0 2014
1 17917
2 14849
3 13303
4 14013
5 12814
6 14541
7 13170
8 14932
9 14368
الف 54
ب 1487
ت 644
ث 1
ج 1480
د 1785
ز 1
س 1371
ش 1
ص 1719
ط 1352
ع 421
ق 1578
ل 1214
م 1529
ن 1145
ه‍ 1143
و 1430
پ 1
ژ (معلولین و جانبازان) 23
ی 1056


Preparing Training Data for CNN

In [ ]:
# Label Encoding
classes = np.unique(y_train)

class_to_index = {
    label: index
    for index, label in enumerate(classes)
}
y_train_encoded = np.array([
    class_to_index[label]
    for label in y_train
])
# Add channel dimension
X_train = np.expand_dims(X_train, axis=-1)

Prepare Validation Data

In [66]:
validation_path = Path("../data/validation")

validation_xml_files = sorted(validation_path.glob("*.xml"))

X_validation = []
y_validation = []

for xml_path in validation_xml_files:

    image_path = xml_path.with_suffix(".jpg")

    characters = process_plate(
        str(image_path),
        str(xml_path)
    )

    for label, image in characters:
        X_validation.append(image)
        y_validation.append(label)

Convert into numpy array

In [67]:
X_validation = np.array(X_validation)
y_validation = np.array(y_validation)

Label encoding for validation data

In [68]:
y_validation_encoded = np.array([
    class_to_index[label]
    for label in y_validation
])

Channel Dimension

In [69]:
X_validation = np.expand_dims(
    X_validation,
    axis=-1
)

In [70]:
print("X_validation shape:", X_validation.shape)
print("y_validation shape:", y_validation_encoded.shape)

X_validation shape: (21852, 64, 32, 1)
y_validation shape: (21852,)


Prepare Test Data

In [71]:
test_path = Path("../data/test")

test_xml_files = sorted(
    test_path.glob("*.xml")
)

X_test = []
y_test = []

for xml_path in test_xml_files:

    image_path = xml_path.with_suffix(".jpg")

    characters = process_plate(
        str(image_path),
        str(xml_path)
    )

    for label, image in characters:
        X_test.append(image)
        y_test.append(label)

Convert test data into numpy array

In [72]:
X_test = np.array(X_test)
y_test = np.array(y_test)

Label encoding test data

In [73]:
y_test_encoded = np.array([
    class_to_index[label]
    for label in y_test
])

Channel Dimension for test data

In [74]:
for xml_path in xml_files:

    tree = ET.parse(xml_path)
    root = tree.getroot()

    for obj in root.findall("object"):

        label = obj.find("name").text

        if label == "پ":
            print("Found in:", xml_path)

            box = obj.find("bndbox")

            xmin = int(box.find("xmin").text)
            ymin = int(box.find("ymin").text)
            xmax = int(box.find("xmax").text)
            ymax = int(box.find("ymax").text)

            print(
                "Bounding box:",
                xmin, ymin, xmax, ymax
            )

Found in: ../data/train/00006.xml
Bounding box: 134 57 200 153


In [75]:
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test_encoded.shape)

X_test shape: (43444, 64, 32)
y_test shape: (43444,)


Creating a CNN model

In [77]:
model = keras.Sequential([
    layers.Input(shape=(64, 32, 1)),

    layers.Conv2D(
        filters=32,
        kernel_size=(3, 3),
        activation="relu",
        padding="same"
    ),

    layers.MaxPooling2D(
        pool_size=(2, 2)
    ),

    layers.Conv2D(
        filters=64,
        kernel_size=(3, 3),
        activation="relu",
        padding="same"
    ),

    layers.MaxPooling2D(
        pool_size=(2, 2)
    ),

    layers.Flatten(),

    layers.Dense(
        units=64,
        activation="relu"
    ),

    layers.Dense(
        units=len(classes),
        activation="softmax"
    )
])

In [78]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 64, 32, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 8, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       524,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 31)             │         2,015 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 545,183 (2.08 MB)

 Trainable params: 545,183 (2.08 MB)

 Non-trainable params: 0 (0.00 B)